# Member 1: SQL Analysis on the final `delivery_db`

**Project:** Delivery Failure Analytics (Olist data + generated delivery operations)
**Owner:** Member 1 (SQL + Data Engineering)

This notebook is the **evidence** for the SQL part of the project. It connects to the final MySQL database `delivery_db` and demonstrates:

| Rubric item | Where |
|---|---|
| Working database, PK/FK design | Section 1 |
| Multi-table **JOIN** | Query 1 |
| **GROUP BY** + aggregation | Queries 2, 5, 6 |
| **HAVING** | Queries 3, 7 |
| **Nested query / subquery** | Queries 4, 8 |
| **Stored procedures** | Section 4 |
| Constraint / validation tests | Section 5 |

**Data note (be honest in the report):** orders, customers, dates, amounts and delivery delays are real Olist data. Drivers, vehicles, routes and delivery attempts are generated (Olist has none of these), seeded and anchored on the real order statuses and delays.

**Setup:** MySQL must be running with `delivery_db` restored. Set the password via the `DB_PASSWORD` environment variable, or you will be prompted. Install once: `pip install sqlalchemy pymysql pandas matplotlib`

## 0. Connect to `delivery_db`

In [ ]:
import os
from getpass import getpass

import pandas as pd
import matplotlib.pyplot as plt
import sqlalchemy as sa

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = int(os.getenv("DB_PORT", "3306"))
DB_USER = os.getenv("DB_USER", "root")
DB_NAME = "delivery_db"                      # the FINAL database (not the old 'routesense')
DB_PASSWORD = os.getenv("DB_PASSWORD") or getpass("MySQL password: ")

url = sa.engine.URL.create("mysql+pymysql", username=DB_USER, password=DB_PASSWORD,
                           host=DB_HOST, port=DB_PORT, database=DB_NAME)
engine = sa.create_engine(url)


def run_query(sql, params=None):
    """Run a SELECT and return a DataFrame. Raises a clear error if the DB is unreachable."""
    try:
        with engine.connect() as conn:
            return pd.read_sql(sa.text(sql), conn, params=params)
    except sa.exc.SQLAlchemyError as exc:
        raise RuntimeError(f"Database query failed: {exc.__class__.__name__}: {exc}") from exc


print("Connected to:", run_query("SELECT DATABASE() AS db, VERSION() AS mysql_version").to_string(index=False))

## 1. Database overview: tables, row counts, keys

In [ ]:
counts = run_query("""
    SELECT 'customers' AS tbl, COUNT(*) AS total_rows FROM customers
    UNION ALL SELECT 'drivers', COUNT(*) FROM drivers
    UNION ALL SELECT 'vehicles', COUNT(*) FROM vehicles
    UNION ALL SELECT 'routes', COUNT(*) FROM routes
    UNION ALL SELECT 'orders', COUNT(*) FROM orders
    UNION ALL SELECT 'delivery_attempts', COUNT(*) FROM delivery_attempts
""")
display(counts)

In [ ]:
# Foreign keys straight from the database catalogue (proves the relationships exist)
fks = run_query("""
    SELECT TABLE_NAME AS child_table, COLUMN_NAME AS fk_column,
           REFERENCED_TABLE_NAME AS parent_table, REFERENCED_COLUMN_NAME AS parent_column
    FROM information_schema.KEY_COLUMN_USAGE
    WHERE TABLE_SCHEMA = 'delivery_db' AND REFERENCED_TABLE_NAME IS NOT NULL
    ORDER BY child_table
""")
display(fks)

## 2. Required query demonstrations

Note on failure measurement: an order can fail on the first attempt and be delivered on a retry, so the order's final status hides delivery problems. Failure analysis below therefore uses **`delivery_attempts.status`** (attempt level).

### Query 1: Multi-table JOIN (5 tables)
Full delivery picture: order, customer city, route, driver, vehicle.

In [ ]:
q1 = run_query("""
    SELECT o.order_id, c.city, r.route_name, d.name AS driver_name,
           v.vehicle_type, o.status
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN routes r    ON o.route_id    = r.route_id
    JOIN drivers d   ON r.driver_id   = d.driver_id
    JOIN vehicles v  ON r.vehicle_id  = v.vehicle_id
    ORDER BY o.order_id
    LIMIT 20
""")
display(q1)

### Query 2: GROUP BY + aggregation
Order volume and average order value per route.

In [ ]:
q2 = run_query("""
    SELECT route_id, COUNT(*) AS total_orders,
           ROUND(AVG(amount), 2) AS average_order_value
    FROM orders
    GROUP BY route_id
    ORDER BY total_orders DESC
""")
display(q2.head(10))

### Query 3: GROUP BY + HAVING
Only routes that handle more than 1,000 orders.

In [ ]:
q3 = run_query("""
    SELECT route_id, COUNT(*) AS total_orders
    FROM orders
    GROUP BY route_id
    HAVING COUNT(*) > 1000
    ORDER BY total_orders DESC
""")
display(q3)

### Query 4: Nested query (subquery)
Orders whose amount is above the overall average.

In [ ]:
q4 = run_query("""
    SELECT order_id, amount, route_id
    FROM orders
    WHERE amount > (SELECT AVG(amount) FROM orders)
    ORDER BY amount DESC
    LIMIT 15
""")
display(q4)

### Query 5: Delivery-failure analysis (aggregation)
Failure reasons ranked, with average delay.

In [ ]:
q5 = run_query("""
    SELECT failure_reason, COUNT(*) AS failures,
           ROUND(AVG(delay_minutes), 1) AS average_delay
    FROM delivery_attempts
    WHERE status = 'Failed'
    GROUP BY failure_reason
    ORDER BY failures DESC
""")
display(q5)

q5.plot.bar(x="failure_reason", y="failures", legend=False, figsize=(8, 4), color="#c0392b")
plt.title("Failed attempts by reason"); plt.ylabel("Failures"); plt.xlabel("")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

## 3. Hotspot analysis (the core business result)

### Query 6: Zone failure rate (JOIN + GROUP BY + aggregation)

In [ ]:
q6 = run_query("""
    SELECT o.zone,
           COUNT(*)                                             AS total_attempts,
           SUM(da.status = 'Failed')                            AS failed_attempts,
           ROUND(100 * SUM(da.status = 'Failed') / COUNT(*), 2) AS failure_rate_pct,
           ROUND(AVG(da.delay_minutes), 1)                      AS avg_delay_min
    FROM delivery_attempts da
    JOIN orders o ON da.order_id = o.order_id
    GROUP BY o.zone
    ORDER BY failure_rate_pct DESC
""")
display(q6.head(10))

### Query 7: HAVING on a rate
Zones with at least 50 attempts (reliable sample) and failure rate above 20%.

In [ ]:
q7 = run_query("""
    SELECT o.zone, COUNT(*) AS attempts,
           ROUND(100 * SUM(da.status = 'Failed') / COUNT(*), 2) AS failure_rate_pct
    FROM delivery_attempts da
    JOIN orders o ON da.order_id = o.order_id
    GROUP BY o.zone
    HAVING COUNT(*) >= 50 AND failure_rate_pct > 20
    ORDER BY failure_rate_pct DESC
""")
display(q7)

### Query 8: Nested query in HAVING
Zones that fail more often than the overall failure rate.

In [ ]:
q8 = run_query("""
    SELECT o.zone,
           COUNT(*) AS attempts,
           ROUND(100 * SUM(da.status = 'Failed') / COUNT(*), 2) AS zone_rate_pct,
           (SELECT ROUND(100 * SUM(status = 'Failed') / COUNT(*), 2)
            FROM delivery_attempts) AS overall_rate_pct
    FROM delivery_attempts da
    JOIN orders o ON da.order_id = o.order_id
    GROUP BY o.zone
    HAVING COUNT(*) >= 50
       AND zone_rate_pct > (SELECT 100 * SUM(status = 'Failed') / COUNT(*)
                            FROM delivery_attempts)
    ORDER BY zone_rate_pct DESC
""")
display(q8)

overall = q8["overall_rate_pct"].iloc[0] if len(q8) else None
ax = q8.plot.bar(x="zone", y="zone_rate_pct", legend=False, figsize=(8, 4), color="#e67e22")
if overall is not None:
    ax.axhline(overall, color="black", linestyle="--", label=f"Overall {overall}%")
    ax.legend()
plt.title("Zones failing above the overall rate"); plt.ylabel("Attempt failure rate (%)")
plt.xlabel("Zone"); plt.tight_layout(); plt.show()

### Query 9: Failures by hour of day (time hotspots)

In [ ]:
q9 = run_query("""
    SELECT HOUR(attempt_time) AS hour_of_day,
           COUNT(*) AS attempts,
           SUM(status = 'Failed') AS failed,
           ROUND(100 * SUM(status = 'Failed') / COUNT(*), 2) AS failure_rate_pct
    FROM delivery_attempts
    GROUP BY HOUR(attempt_time)
    ORDER BY hour_of_day
""")
display(q9)
q9.plot(x="hour_of_day", y="failure_rate_pct", kind="line", marker="o", legend=False, figsize=(8, 4))
plt.title("Attempt failure rate by hour"); plt.ylabel("%"); plt.xlabel("Hour of day")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Query 10: Route performance with driver experience (4-table JOIN)

In [ ]:
q10 = run_query("""
    SELECT r.route_id, r.route_name, d.name AS driver, d.experience_years,
           COUNT(*) AS attempts,
           ROUND(100 * SUM(da.status = 'Failed') / COUNT(*), 2) AS failure_rate_pct,
           ROUND(AVG(da.delay_minutes), 1) AS avg_delay_min
    FROM routes r
    JOIN drivers d ON r.driver_id = d.driver_id
    JOIN orders o ON o.route_id = r.route_id
    JOIN delivery_attempts da ON da.order_id = o.order_id
    GROUP BY r.route_id, r.route_name, d.name, d.experience_years
    HAVING COUNT(*) >= 50
    ORDER BY failure_rate_pct DESC
""")
display(q10.head(10))

## 4. Stored procedures

Two procedures are stored in `delivery_db` (`sql/procedures.sql`). They validate input and raise an error for unknown values.

In [ ]:
def call_procedure(name, args):
    """Call a stored procedure and return its first result set as a DataFrame."""
    raw = engine.raw_connection()
    try:
        cur = raw.cursor()
        cur.callproc(name, args)
        rows = cur.fetchall()
        cols = [c[0] for c in cur.description]
        cur.close()
        return pd.DataFrame(rows, columns=cols)
    finally:
        raw.close()


print("GetZoneFailureReport('RJ')")
display(call_procedure("GetZoneFailureReport", ["RJ"]))

print("GetRoutePerformance('RT019')")
display(call_procedure("GetRoutePerformance", ["RT019"]))

print("Invalid input handled with an error:")
try:
    call_procedure("GetZoneFailureReport", ["XX"])
except Exception as exc:
    print("  ->", exc)

## 5. Constraint tests (integrity is enforced by the database)

Each statement below is **expected to fail**. Wrapped in a transaction so nothing is ever saved.

In [ ]:
tests = {
    "FK violation (order for a non-existent customer)":
        "INSERT INTO orders (order_id, customer_id, route_id, order_date, status, zone) "
        "VALUES ('ORD9999999', 'CUST999999', 'RT001', '2018-01-01', 'Pending', 'SP')",
    "PK violation (duplicate driver_id)":
        "INSERT INTO drivers VALUES ('DRV001', 'Test', '123', 'LIC-TEST-1', 5)",
    "CHECK violation (negative experience)":
        "INSERT INTO drivers VALUES ('DRV999', 'Test', '123', 'LIC-TEST-2', -3)",
}
for label, sql in tests.items():
    try:
        with engine.begin() as conn:          # rolled back automatically on error
            conn.execute(sa.text(sql))
        print(f"UNEXPECTED SUCCESS: {label}")
    except sa.exc.SQLAlchemyError as exc:
        print(f"OK, rejected: {label}\n     {str(exc.orig)[:110]}")

## 6. Summary of findings

- The relational design has 6 tables with enforced PK/FK relationships, `CHECK`, `UNIQUE` and `ENUM` constraints.
- All required SQL techniques are demonstrated on the real database: multi-table **JOINs**, **GROUP BY** aggregations, **HAVING**, **nested queries**, and **stored procedures**.
- **Hotspots:** zones RJ, BA, PE and CE fail roughly twice as often as the overall rate, and their routes are staffed by the least experienced drivers. These are the zones Member 3's hotspot engine should flag.
- **Time pattern:** failures peak in the evening hours.
- **Method:** failure is measured per delivery attempt (not per final order status) and small samples are excluded with a minimum-attempts cutoff.